In [0]:
%pip install lightgbm

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
import lightgbm

import mlflow
import mlflow.lightgbm

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

print("Libraries loaded successfully")

Libraries loaded successfully


In [0]:
DATA_PATH = "demand_forecasting.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))




Rows: 76000
Columns: 16


In [0]:
df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

print("Date conversion and sorting completed.")

Date conversion and sorting completed.


In [0]:
GROUP_COLS = ["Store ID","Product ID"]

print("Groups:", GROUP_COLS)

Groups: ['Store ID', 'Product ID']


In [0]:
df["day_of_week"] = df["Date"].dt.dayofweek

df["day_of_month"] = df["Date"].dt.day

df["week_of_year"] = (
    df["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

df["month"] = df["Date"].dt.month

df["quarter"] = df["Date"].dt.quarter

df["year"] = df["Date"].dt.year

df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)


In [0]:
df["lag_1"] = (df.groupby(GROUP_COLS)["Demand"].shift(1))

df["lag_7"] = (df.groupby(GROUP_COLS)["Demand"].shift(7))

df["lag_14"] = (df.groupby(GROUP_COLS)["Demand"].shift(14))

df["lag_28"] = (df.groupby(GROUP_COLS)["Demand"].shift(28))

In [0]:
df["rolling_mean_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(7)
           .mean()
      )
)

df["rolling_mean_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(14)
           .mean()
      )
)

df["rolling_mean_28"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(28)
           .mean()
      )
)


In [0]:
df["rolling_std_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(7)
           .std()
      )
)

df["rolling_std_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(14)
           .std()
      )
)

In [0]:
HISTORY_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14"
]

df_model = df.dropna(
    subset=HISTORY_FEATURES
).copy()

print("Original rows:", len(df))
print("Rows after feature engineering:", len(df_model))

Original rows: 76000
Rows after feature engineering: 73200


In [0]:
LAG_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

ROLLING_FEATURES = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14"
]

BUSINESS_FEATURES = [
    "Inventory Level",
    "Units Ordered",
    "Price",
    "Discount",
    "Promotion",
    "Competitor Pricing"
]

TIME_FEATURES = [
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend"
]

CATEGORICAL_FEATURES = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality",
    "Epidemic"
]

FEATURES = (
    LAG_FEATURES
    + ROLLING_FEATURES
    + BUSINESS_FEATURES
    + TIME_FEATURES
    + CATEGORICAL_FEATURES
)

TARGET = "Demand"

print("Number of features:", len(FEATURES))

print("\nFeatures:")
for feature in FEATURES:
    print("-", feature)

print("\nTarget:", TARGET)

Number of features: 29

Features:
- lag_1
- lag_7
- lag_14
- lag_28
- rolling_mean_7
- rolling_mean_14
- rolling_mean_28
- rolling_std_7
- rolling_std_14
- Inventory Level
- Units Ordered
- Price
- Discount
- Promotion
- Competitor Pricing
- day_of_week
- day_of_month
- week_of_year
- month
- quarter
- year
- is_weekend
- Store ID
- Product ID
- Category
- Region
- Weather Condition
- Seasonality
- Epidemic

Target: Demand


In [0]:
missing_features = [
    feature
    for feature in FEATURES
    if feature not in df_model.columns
]

if missing_features:

    raise ValueError(
        f"Missing features: {missing_features}"
    )

else:

    print("All features are available.")

All features are available.


In [0]:
dates = sorted(df_model["Date"].unique())

train_end = dates[int(len(dates) * 0.70)]

valid_end = dates[int(len(dates) * 0.85)]

print("Train end:", train_end)
print("Validation end:", valid_end)

Train end: 2023-06-25 00:00:00
Validation end: 2023-10-13 00:00:00


In [0]:
train_df = df_model[df_model["Date"] <= train_end].copy()

In [0]:
valid_df = df_model[(df_model["Date"] > train_end) & (df_model["Date"] <= valid_end)].copy()

In [0]:
test_df = df_model[df_model["Date"] > valid_end].copy()

In [0]:
print("DATA SPLIT")
print("================================")

print("Train      :", train_df.shape)
print("Validation :", valid_df.shape)
print("Test       :", test_df.shape)

print("================================")

DATA SPLIT
Train      : (51300, 32)
Validation : (11000, 32)
Test       : (10900, 32)


In [0]:
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_valid = valid_df[FEATURES].copy()
y_valid = valid_df[TARGET].copy()

X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (51300, 29)
y_train: (51300,)
X_valid: (11000, 29)
y_valid: (11000,)
X_test: (10900, 29)
y_test: (10900,)


In [0]:
for col in CATEGORICAL_FEATURES:

    X_train[col] = (X_train[col].astype("category"))

    X_valid[col] = (X_valid[col].astype("category"))

    X_test[col] = (X_test[col].astype("category"))

print("Categorical columns converted.")


Categorical columns converted.


In [0]:
EXPERIMENT_NAME = ("/Shared/demand-forecasting")

mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow experiment:",EXPERIMENT_NAME)


MLflow experiment: /Shared/demand-forecasting


In [0]:
MODEL_PARAMS = {
    "objective": "regression",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

print("Model parameters:")

for key, value in MODEL_PARAMS.items():
    print(f"{key}: {value}")

Model parameters:
objective: regression
n_estimators: 500
learning_rate: 0.05
num_leaves: 31
max_depth: -1
subsample: 0.8
colsample_bytree: 0.8
random_state: 42


In [0]:
from mlflow.models import infer_signature

In [0]:
with mlflow.start_run(run_name="lightgbm_demand_forecasting"):

    # Create model
    model = LGBMRegressor(**MODEL_PARAMS)

    # Train model
    model.fit(X_train,y_train,categorical_feature=CATEGORICAL_FEATURES)
    signature = infer_signature(X_train,model.predict(X_train))
    
    input_example = X_train.iloc[[0]]

    mlflow.lightgbm.log_model(
        model,
        name="model",
        signature=signature,
        input_example=input_example
        )

    print("Model trained successfully.")


    # --------------------------------------------------------
    # Validation predictions
    # --------------------------------------------------------

    valid_predictions = model.predict(X_valid)

    # Demand should not be negative
    valid_predictions = np.maximum(valid_predictions,0)


    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------

    validation_mae = mean_absolute_error(y_valid,valid_predictions)

    validation_rmse = np.sqrt(mean_squared_error(y_valid,valid_predictions))

    validation_r2 = r2_score(y_valid,valid_predictions)


    # Safe MAPE
    mask = y_valid != 0

    validation_mape = (
        np.mean(
            np.abs(
                (
                    y_valid[mask]
                    -
                    valid_predictions[mask]
                )
                /
                y_valid[mask]
            )
        )
        * 100
    )


    # --------------------------------------------------------
    # Log parameters
    # --------------------------------------------------------

    mlflow.log_params(MODEL_PARAMS)

    mlflow.log_param("target",TARGET)

    mlflow.log_param("num_features",len(FEATURES))

    mlflow.log_param("split_type","time_based")


    # --------------------------------------------------------
    # Log metrics
    # --------------------------------------------------------

    mlflow.log_metric("validation_mae",validation_mae)

    mlflow.log_metric("validation_rmse",validation_rmse)

    mlflow.log_metric("validation_r2",validation_r2)

    mlflow.log_metric("validation_mape",validation_mape)


    # --------------------------------------------------------
    # Log model
    # --------------------------------------------------------

    mlflow.lightgbm.log_model(
        model,
        "model",
        signature=signature,
        input_example=input_example
        )

    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print("")
    print("================================")
    print("VALIDATION RESULTS")
    print("================================")

    print("MAE  :",validation_mae)

    print("RMSE :",validation_rmse)

    print("R2   :",validation_r2)

    print("MAPE :",validation_mape)

    print("================================")

2026/09/05 10:27:26 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o11.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumented

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005708 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3489
[LightGBM] [Info] Number of data points in the train set: 51300, number of used features: 29
[LightGBM] [Info] Start training from score 103.155127


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/05 10:27:39 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o21.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.i

Model trained successfully.


2026/09/05 10:27:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/05 10:27:52 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o52.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand


VALIDATION RESULTS
MAE  : 11.92655913822473
RMSE : 16.849697084376338
R2   : 0.8793130435926209
MAPE : 15.188812339497115


In [0]:
baseline_predictions = (valid_df["lag_1"].values)

baseline_predictions = np.maximum(baseline_predictions,0)

baseline_mae = mean_absolute_error(y_valid,baseline_predictions)

baseline_rmse = np.sqrt(mean_squared_error(y_valid,baseline_predictions))

print("================================")
print("BASELINE RESULTS")
print("================================")

print("Baseline MAE :",baseline_mae)

print("Baseline RMSE:",baseline_rmse)

print("================================")


BASELINE RESULTS
Baseline MAE : 44.34309090909091
Baseline RMSE: 57.66052691084572


In [0]:
print("================================")
print("MODEL COMPARISON")
print("================================")

print("Baseline MAE :",baseline_mae)

print("LightGBM MAE  :",validation_mae)

print("Baseline RMSE:",baseline_rmse)

print("LightGBM RMSE :",validation_rmse)

print("================================")


if validation_mae < baseline_mae:

    print("LightGBM improved over the baseline.")

else:

    print("LightGBM did NOT improve over the baseline.")

MODEL COMPARISON
Baseline MAE : 44.34309090909091
LightGBM MAE  : 11.92655913822473
Baseline RMSE: 57.66052691084572
LightGBM RMSE : 16.849697084376338
LightGBM improved over the baseline.


In [0]:
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
})

importance = importance.sort_values("importance",ascending=False)

display(importance.head(20))

feature,importance
Price,3908
Product ID,2490
Units Ordered,1337
Competitor Pricing,1266
Store ID,1137
Category,963
Inventory Level,603
rolling_mean_7,596
Discount,548
Epidemic,340


In [0]:
test_predictions = model.predict(X_test)

test_predictions = np.maximum(test_predictions,0)

test_mae = mean_absolute_error(y_test,test_predictions)

test_rmse = np.sqrt(mean_squared_error(y_test,test_predictions))

test_r2 = r2_score(y_test,test_predictions)

test_mask = y_test != 0

test_mape = (
    np.mean(
        np.abs(
            (
                y_test[test_mask]
                -
                test_predictions[test_mask]
            )
            /
            y_test[test_mask]
        )
    )
    * 100
)


print("================================")
print("FINAL TEST RESULTS")
print("================================")

print("Test MAE  :",test_mae)

print("Test RMSE :",test_rmse)

print("Test R2   :",test_r2)

print("Test MAPE :",test_mape)

print("================================")


FINAL TEST RESULTS
Test MAE  : 11.123409847455939
Test RMSE : 15.254303021916487
Test R2   : 0.8835180158362557
Test MAPE : 17.587166719847858


In [0]:
print("MODEL TRAINING COMPLETED SUCCESSFULLY")
print("============================================")

print("Model      : LightGBM")
print("Experiment :", EXPERIMENT_NAME)

print("")
print("Validation MAE :", validation_mae)
print("Validation RMSE:", validation_rmse)

print("")
print("Test MAE       :", test_mae)
print("Test RMSE      :", test_rmse)

print("============================================")

MODEL TRAINING COMPLETED SUCCESSFULLY
Model      : LightGBM
Experiment : /Shared/demand-forecasting

Validation MAE : 11.92655913822473
Validation RMSE: 16.849697084376338

Test MAE       : 11.123409847455939
Test RMSE      : 15.254303021916487


In [0]:
display(dbutils.fs.ls("/Workspace/Users/darshanpatil6968@gmail.com/demand_forecast/"))



path,name,size,modificationTime
dbfs:/Workspace/Users/darshanpatil6968@gmail.com/demand_forecast/GPTforecast.ipynb,GPTforecast.ipynb,51565,1788526326460
dbfs:/Workspace/Users/darshanpatil6968@gmail.com/demand_forecast/cleaned_demand_forecasting_data.csv,cleaned_demand_forecasting_data.csv,10204341,1787574057541
dbfs:/Workspace/Users/darshanpatil6968@gmail.com/demand_forecast/demand-forecast/,demand-forecast/,4096,1788604085921
dbfs:/Workspace/Users/darshanpatil6968@gmail.com/demand_forecast/demand_forecasting.csv,demand_forecasting.csv,6270401,1787664115381
dbfs:/Workspace/Users/darshanpatil6968@gmail.com/demand_forecast/demant_forecast.ipynb,demant_forecast.ipynb,17832,1788507344909


In [0]:
display(dbutils.fs.ls("/Volumes/demand_catalog/default/demand_volume"))

path,name,size,modificationTime
dbfs:/Volumes/demand_catalog/default/demand_volume/demand_forecasting.csv,demand_forecasting.csv,6270401,1788526465000


In [0]:
DATA_PATH = "/Volumes/demand_catalog/default/demand_volume/demand_forecasting.csv"

df_test = pd.read_csv(DATA_PATH)

print("CSV loaded successfully!")
print("Rows:", len(df_test))
print("Columns:", len(df_test.columns))

display(df_test.head())

CSV loaded successfully!
Rows: 76000
Columns: 16


Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59
